![HydroCycle](images/hydro_5cycle.jpg)

# Retrieve and Analyze Hydrology data for a watershed of interest

To make predictions for reservoir operations, water supply, flood control, etc, we need to collect data to train/calibrate hydrologic models. This includes streamflow, current environmental conditions (e.g., snow water equivalent), and future weather predictions. This exercise will build on the previous SNOTEL module to work towards building a hydrologic module.

Need to find a station? Use the [USGS NWIS mapper system](https://apps.usgs.gov/nwismapper/)


Click the link and explore!

## Create a HydroDataFrame

We want to create one data frame containing streamflow, meteological information, and SNOTEL for our period of record

In [1]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = 'files/SNOTEL'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
    



In [ ]:
#The TES site is missing many values and will not be useful for our analysis, remove it
unprocessed_SNOTEL.pop('TES', None)

#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)

SNOTEL_df.head()

In [ ]:
met_df.head()

In [ ]:
cleaned.head()

In [ ]:
#find the latest start date and the earliest end date for SNOTEL_df, met_df, cleaned
begin_date = max([df.index.min() for df in [SNOTEL_df, met_df, cleaned]])
end_date = min([df.index.max() for df in [SNOTEL_df, met_df, cleaned]])

#clip each dataframe to have the same begin and end dates
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]
met_df = met_df[(met_df.index >= begin_date) & (met_df.index <= end_date)]
cleaned = cleaned[(cleaned.index >= begin_date) & (cleaned.index <= end_date)]

#merge the SNOTEL_df, met_df, and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, met_df, cleaned], axis=1)
Hydro_df.head(50)

In [ ]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head(50)

In [ ]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

In [ ]:
Hydro_df.tail()

## Data exploration

In [ ]:
#For year 2019, plot all SWE_cm columns
SWE_df = Hydro_df.loc['2019-10-01':'2020-09-30'].copy()

#select all columns with 'SWE_cm' in the name
SWE_df = SWE_df.loc[:, SWE_df.columns.str.contains('SWE_cm')]

#plot
SWE_df.plot(figsize=(10, 6))

In [ ]:
#plot the relationship between SWE_cm and flow_cms
#For year 2019, plot all SWE_cm columns
df = Hydro_df.loc['2019-10-01':'2020-09-30'].copy()

#select all columns with 'SWE_cm' and 'flow_cms' in the name
df = df.loc[:, df.columns.str.contains('SWE_cm') | df.columns.str.contains('flow_cms')]

#get colum names that contain SWE_cm
swe_cols = df.columns[df.columns.str.contains('SWE_cm')]

#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('SWE (cm)', color='darkorange', fontsize=12, fontweight='bold')
for swe in swe_cols:    
    ax1.plot(df.index, df[swe], linewidth=2, label = swe)

ax1.plot(df.index, df.flow_cms, color='blue', linewidth=2, label='Streamflow') 
ax1.tick_params(axis='y', labelcolor='darkorange')
ax1.grid(True, alpha=0.3)
#make second axis for streamflow
ax2 = ax1.twinx()
ax2.set_ylabel('Streamflow (cms)', color='blue', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='blue')

#show a legend
ax1.legend()

# Title and Layout
plt.title('SSWE and Streamflow for water year 2019 at USGS gage: ' + usgs_gage_id, fontsize=14, pad=15)
fig.tight_layout()
plt.show()


In [ ]:
df.columns

In [ ]:
#plot the relationship between SWE_cm and temperature
#For year 2019, plot all SWE_cm columns
df = Hydro_df.loc['2019-10-01':'2020-09-30'].copy()

#get colum names that contain SWE_cm
swe_cols = df.columns[df.columns.str.contains('SWE_cm')]

#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('SWE (cm)', color='darkblue', fontsize=12, fontweight='bold')
#for swe in swe_cols:    
ax1.plot(df.index, df.TUM_SWE_cm, linewidth=2, label = 'SWE', color='darkblue')
ax1.plot(df.index, df.tmax_C, color='red', linewidth=2)

ax1.tick_params(axis='y', labelcolor='darkblue')
ax1.grid(True, alpha=0.3)
#make second axis for streamflow
ax2 = ax1.twinx()
ax2.set_ylabel('Temperature (°C)', color='red', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='red')

#show a legend
ax1.legend()

# Title and Layout
plt.title('Snow and Temperature for water year 2019 at USGS gage: ' + usgs_gage_id, fontsize=14, pad=15)
fig.tight_layout()
plt.show()


In [ ]:
Hydro_df.columns

In [ ]:
#plot the relationship between streamflow and precipitation
#For year 2019, plot all SWE_cm columns
df = Hydro_df.loc['2019-10-01':'2020-09-30'].copy()

#get colum names that contain SWE_cm
swe_cols = df.columns[df.columns.str.contains('SWE_cm')]

#make the plot
fig, ax1 = plt.subplots(figsize=(6, 6))

# --- Primary Y-axis: SWE_cm ---
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Precipitation (mm)', color='darkblue', fontsize=12, fontweight='bold')
#for swe in swe_cols:    
ax1.plot(df.index, df.prcp_mm_day, linewidth=2, label = 'Precipitation', color='darkblue')
ax1.plot(df.index, df.flow_cms, color='orange', linewidth=2)

ax1.tick_params(axis='y', labelcolor='darkblue')
ax1.grid(True, alpha=0.3)
#make second axis for streamflow
ax2 = ax1.twinx()
ax2.set_ylabel('Streamflow (cms)', color='orange', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='orange')

#show a legend
ax1.legend()

# Title and Layout
plt.title('Precipitation and Streamflow for water year 2019 at USGS gage: ' + usgs_gage_id, fontsize=14, pad=15)
fig.tight_layout()
plt.show()
